<a href="https://colab.research.google.com/github/seleneyong/ISYS2001-MyWork/blob/main/Copy_of_guided_pandas_worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2007%20-%20Directing%20Pandas/guided_pandas_worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with tables: from your own loop to pandas

Last week you organised information with dictionaries, and along the way you wrote a
loop that added up amounts by category. That loop was honest work: you could read
every line of it and explain exactly what it did.

This week we meet a tool called pandas that does the same kind of job on tables of
data. The point of today is not to abandon what you already know. It is the opposite.
We are going to write the totals-by-category calculation the way you already
understand it, then watch pandas do the very same thing, and check that the two
agree. Once you have seen that pandas is doing work you could already do by hand,
you can reach for it knowing what it is actually doing underneath.

This is a guided worksheet. We will run each cell together and talk through it as we
go. Keep your attention on *what pandas is replacing* and *why you would choose it*,
not on memorising the method names.

## The data

We have a small file of transactions: a date, an amount, a category, and a short
description for each one. Let us start by just looking at it as plain text, the way
you would open any file, before we bring in any new tools.

### First, get the file into Colab

Opening this notebook from the badge brings the notebook across, but not the data
file that sits beside it in the repository. So before we run anything, we need to put
`transactions.csv` into this Colab session by hand.

In the panel on the left, click the folder icon to open the file browser, then click
the upload button and choose `transactions.csv`. It will appear in the working folder,
which is exactly where the code below looks for it.

You only need to do this once per session. If Colab times out and starts a fresh
session, the file disappears and you upload it again.

In [6]:
with open('transactions.csv') as f:
    for line in f:
        print(line.strip())

Date,Amount,Category,Description
2024-08-01,$45.50,Groceries,Woolworths Weekly Shop
2024-08-02,$12.00,Transport,Opal Card Top-up
2024-08-03,$89.95,Entertainment,Concert Tickets - Enmore Theatre
2024-08-04,$3.50,Coffee,Campus Cafe Flat White
2024-08-05,$120.00,Groceries,Coles Weekly Shop
2024-08-06,-$25.00,Refund,Returned Textbook - Co-op Bookshop
2024-08-07,$85.00,Dining,Birthday Dinner - The Rocks
2024-08-08,$15.95,Coffee,Starbucks Double Shot
2024-08-09,$67.80,Utilities,Electricity Bill - Origin Energy
2024-08-10,$4.20,Coffee,Campus Cafe Cappuccino
2024-08-11,$32.50,Transport,Uber to Airport
2024-08-12,$156.00,Groceries,Woolworths Fortnightly Shop
2024-08-13,$8.50,Coffee,Gloria Jeans Large Latte
2024-08-14,$95.00,Entertainment,Movies and Dinner - Event Cinemas
2024-08-15,$28.90,Dining,Lunch - Guzman y Gomez
2024-08-16,-$12.50,Refund,Coffee Shop Refund
2024-08-17,$45.00,Transport,Petrol - Caltex Woolworths
2024-08-18,$73.20,Groceries,IGA Local Shop
2024-08-19,$22.00,Entertainment,Netf

Notice two things while we look at it. The amounts have dollar signs in front of
them, so they are text, not numbers yet. And a couple of the amounts are negative:
those are refunds. Both of those facts will matter in a moment.

## The question

Here is the job, in plain words:

> For each category, add up how much was spent, and show the categories from the
> highest total to the lowest.

That is exactly the sort of thing you did last week. Let us write it that way first.

## First, the way you already know

We will read the file, build a dictionary of running totals by category, and sort it.
There is nothing new here. Read it and satisfy yourself that you could have written
it.

In [7]:
import csv

category_totals = {}

with open('transactions.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # The amount is text like '$45.50' or '-$25.00', so strip the dollar
        # sign and turn it into a number before we add it up.
        amount_text = row['Amount'].replace('$', '')
        amount = float(amount_text)

        category = row['Category']
        if category in category_totals:
            category_totals[category] += amount
        else:
            category_totals[category] = amount

# Sort the categories from highest total to lowest.
ordered = sorted(category_totals.items(), key=lambda pair: pair[1], reverse=True)

for category, total in ordered:
    print(f"{category:<15} {total:>8.2f}")

Groceries         394.70
Entertainment     206.95
Dining            113.90
Transport          89.50
Utilities          67.80
Coffee             37.95
Refund            -37.50


Take a moment with that output. Every number in it came from a loop you could walk
through by hand. This is our source of truth. Whatever pandas gives us next, it has
to agree with this.

## Now, the same job with pandas

pandas reads a table into something called a DataFrame. Think of it as the whole
spreadsheet held in one variable, with the columns still named the way they are in
the file.

We will do the same three steps as before: read the file, turn the amounts into
numbers, then total by category. Watch how each step lines up with what we just did
by hand.

In [8]:
import pandas as pd

# Step 1: read the whole table in one line.
df = pd.read_csv('transactions.csv')

df

,Date,Amount,Category,Description
0,2024-08-01,$45.50,Groceries,Woolworths Weekly Shop
1,2024-08-02,$12.00,Transport,Opal Card Top-up
2,2024-08-03,$89.95,Entertainment,Concert Tickets - Enmore Theatre
3,2024-08-04,$3.50,Coffee,Campus Cafe Flat White
4,2024-08-05,$120.00,Groceries,Coles Weekly Shop
5,2024-08-06,-$25.00,Refund,Returned Textbook - Co-op Bookshop
6,2024-08-07,$85.00,Dining,Birthday Dinner - The Rocks
7,2024-08-08,$15.95,Coffee,Starbucks Double Shot
8,2024-08-09,$67.80,Utilities,Electricity Bill - Origin Energy
9,2024-08-10,$4.20,Coffee,Campus Cafe Cappuccino


That one line replaced our open-the-file-and-loop. The whole table is now in `df`,
columns and all.

Now the same cleaning we did by hand: strip the dollar sign and turn the column into
numbers . We do it once, to the whole column at once, rather than row by row.

In [9]:
# Step 2: clean the Amount column once, for every row at the same time.
df['Amount'] = df['Amount'].str.replace('$', '', regex=False)
df['Amount'] = pd.to_numeric(df['Amount'])

df['Amount']

,Amount
0,45.50
1,12.00
2,89.95
3,3.50
4,120.00
5,-25.00
6,85.00
7,15.95
8,67.80
9,4.20


And the totalling. Our loop said "for each category, keep a running total". pandas
says the same thing in one line: group the rows by category, then sum the amounts.

In [10]:
# Step 3: total by category, highest first.
category_totals_pandas = df.groupby('Category')['Amount'].sum()
category_totals_pandas = category_totals_pandas.sort_values(ascending=False)

category_totals_pandas

,Amount
Category,
Groceries,394.70
Entertainment,206.95
Dining,113.90
Transport,89.50
Utilities,67.80
Coffee,37.95
Refund,-37.50


## Do they agree?

This is the important cell. If pandas is really doing the same job as our loop, the
numbers should match. Let us not take it on faith. Let us check.

In [ ]:
for category, total in ordered:
    pandas_total = category_totals_pandas[category]
    match = "yes" if round(total, 2) == round(pandas_total, 2) else "NO"
    print(f"{category:<15} hand: {total:>8.2f}   pandas: {pandas_total:>8.2f}   match: {match}")

They agree, category by category. That is the whole point of today. pandas did not
do anything mysterious. It did the calculation you already understand, in three lines
instead of a dozen, and it handled the whole column at once instead of one row at a
time.

That is what makes it worth reaching for: not that it is clever, but that it is the
same idea with less to type and less to get wrong once your tables get large.

## Planning the next step before we ask for help

Suppose we now want the *average* transaction size per category, not the total. Before
we write anything or ask an assistant for anything, we say the plan in plain words:

> Group the rows by category, as before. But instead of adding the amounts up, take
> their average.

That plan is the thing we check the answer against. Let us ask Gemini for the pandas
line, giving it the plan and the constraint that we want to be able to read it:

> *I have a pandas DataFrame `df` with a numeric column `Amount` and a text column
> `Category`. I want the average Amount for each Category, sorted highest to lowest.
> Give me one or two lines of pandas and explain what each part does.*

Whatever it hands back, we read it against our plan: is it grouping by category? Is it
taking a mean and not a sum? Type the result into the cell below yourself, run it, and
make sure it does what the plan said.

In [ ]:
# Write the average-per-category line here, guided by the plan above.
# Check the result against what you expected before moving on.


## When is a dictionary still the right tool?

pandas is not the answer to everything. A dictionary is still the better choice when
you are holding a handful of named settings, or looking one thing up by name, or
building up a small structure as you go. Reach for pandas when you have a *table*:
many rows with the same columns, where you want to summarise, group, or filter across
all of them at once.

Today's job, many transactions summarised by category, is squarely a table job. Last
week's job, designing the shape of a single record, was a dictionary job. Knowing
which one you are holding is the skill worth keeping.